# Async LLM enrichment for flight incident reports

This notebook demonstrates a simple, practical async LLM workflow over a CSV of flight incident reports.

For each incident, it sends **two existing text fields** to the model:

- `Narrative`
- `Result`

The model returns **two new fields**:

- `timeline_text` — a short plain-language sequence of what happened
- `normalized_summary` — a short standardized summary of the incident and outcome

The notebook keeps the structure of the earlier async demo:

1. configuration and setup
2. a **sync baseline**
3. an **async version**
4. timing comparison
5. storing the results back into a dataframe

## What you'll need

- Python 3.9+
- `OPENAI_API_KEY` set in your environment
- the `openai`, `pandas`, and `nest_asyncio` packages installed

Example install command:

```bash
pip install -U openai pandas nest_asyncio
```

In [1]:
import os
import json
import time
import asyncio
from pathlib import Path

import nest_asyncio
import pandas as pd
from openai import OpenAI, AsyncOpenAI

nest_asyncio.apply()

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY is not set in the environment.")

print("API key detected.")

API key detected.


## Configuration

Adjust these values as needed.

- `MAX_ROWS` controls how many incidents you process in the demo.
- `CONCURRENCY_LIMIT` helps avoid pushing too hard into rate limits.

In [5]:
CSV_FILE = "ASRS_DBOnline_2025_11.csv"
HEADER_ROW = 1

MODEL = "gpt-5-mini"
MAX_ROWS = 37
CONCURRENCY_LIMIT = 5

OUTPUT_CSV = "asrs_incidents_enriched.csv"

## Load the source data

This CSV uses a two-row header, so the notebook reads row 2 as the actual column names.

In [4]:
df_raw = pd.read_csv(CSV_FILE, header=HEADER_ROW)
print(f"Loaded {CSV_FILE}")
print(f"Shape: {df_raw.shape}")
print(f"Columns: {len(df_raw.columns)}")

Loaded ASRS_DBOnline_2025_11.csv
Shape: (37, 126)
Columns: 126


## Select the fields for the demo

This keeps the task intentionally simple:

- use the original `Narrative` and `Result` text
- keep `ACN` and `Synopsis` for reference
- drop rows with missing or blank text

In [6]:
base_cols = ["ACN", "Narrative", "Result", "Synopsis"]
work_df = df_raw[base_cols].copy()

for col in ["Narrative", "Result", "Synopsis"]:
    work_df[col] = work_df[col].fillna("").astype(str).str.strip()

mask = (work_df["Narrative"] != "") & (work_df["Result"] != "")
work_df = work_df.loc[mask].reset_index(drop=True)

print(f"Rows with both Narrative and Result present: {len(work_df)}")

demo_df = work_df.head(MAX_ROWS).copy()
print(f"Rows selected for demo: {len(demo_df)}")
demo_df.head(3)

Rows with both Narrative and Result present: 37
Rows selected for demo: 37


,ACN,Narrative,Result,Synopsis
0,2301768,Approaching 100 kts (approximately 95 kts); we...,Air Traffic Control Provided Assistance; Fligh...,A320 Captain reported rejecting a takeoff at 9...
1,2301817,Spot X ZZZ. After pushback. Both engines runn...,Air Traffic Control Provided Assistance; Fligh...,B777 crew reported receiving a cargo fire EICA...
2,2302457,Prior to departure in ZZZ1 we had a fuel confi...,Flight Crew Inflight Shutdown; Flight Crew FLC...,Air carrier Captain reported a fuel leak and s...


## Preview a sample record

In [7]:
sample_idx = 0
print("ACN:", demo_df.loc[sample_idx, "ACN"])
print("\nNARRATIVE:\n")
print(demo_df.loc[sample_idx, "Narrative"][:2000])
print("\nRESULT:\n")
print(demo_df.loc[sample_idx, "Result"][:1000])

ACN: 2301768

NARRATIVE:

Approaching 100 kts (approximately 95 kts); we received a Master Warning and Engine 2 Fire indication. As the reject was initiated; the fire indication disappeared. After completing the Reject maneuver; we referenced the QRC and confirmed with Tower and FAs that they saw no indications of a fire. We then reviewed the Rejected Takeoff checklist in the Flight Manual; exited the runway; and had ARFF (Airport Rescue and Firefighting) look over the aircraft. After ARFF indicted that everything looked normal; we shutdown Engine 2 to taxi back using normal single-engine procedures. After parking and confirming that the aircraft was chocked; the brakes were left off in accordance with flight manual guidance. Throughout the situation; passengers were kept informed and there was no indication of any injuries. After passengers deplaned; the Captain communicated with Dispatch/Maintenance Control/Chief Pilot and the crew debriefed the event. The crew subsequently operated 

## Define the LLM task

The model should not redo the structured self-reported fields. Instead, it should produce two **new normalized text fields** that make the narratives easier to compare later.

The output is kept intentionally small and simple:

- `timeline_text`: a short semicolon-separated timeline
- `normalized_summary`: a short plain-language summary

To make parsing reliable, the prompt asks for strict JSON with just those two keys.

In [8]:
sync_client = OpenAI()
async_client = AsyncOpenAI()

SYSTEM_INSTRUCTIONS = (
    "You are helping normalize aviation incident reports for later analysis. "
    "Be concise, factual, and grounded only in the provided text. "
    "Do not add facts that are not supported by the Narrative or Result fields."
)

JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "timeline_text": {
            "type": "string",
            "description": "Simple semicolon-separated timeline of the main events."
        },
        "normalized_summary": {
            "type": "string",
            "description": "One or two sentences summarizing the incident and outcome in standardized language."
        },
    },
    "required": ["timeline_text", "normalized_summary"],
    "additionalProperties": False,
}

def build_input(narrative: str, result_text: str) -> str:
    return f"""System instructions:
{SYSTEM_INSTRUCTIONS}

Task:
Read the incident fields below and return JSON with exactly two string fields:

1. timeline_text
   - Write a short timeline in simple text.
   - Use semicolons to separate the steps.
   - Focus on the main operational sequence only.
   - Do not include bullet points or numbering.

2. normalized_summary
   - Write a short, standardized summary in simple text.
   - Mention the key event and the outcome.
   - Keep it to 1-2 sentences.

Narrative:
{narrative}

Result:
{result_text}
"""

def parse_json_output(text: str) -> dict:
    data = json.loads(text)
    return {
        "timeline_text": str(data["timeline_text"]).strip(),
        "normalized_summary": str(data["normalized_summary"]).strip(),
    }

## Helper functions

The notebook uses the OpenAI Responses API, which OpenAI recommends for new projects. The Python SDK provides both sync and async clients, and the Responses API includes a convenience `output_text` helper. OpenAI also documents Structured Outputs for schema-constrained JSON, using `text.format` with a JSON schema in the Responses API.

In [9]:
def enrich_one_sync(narrative: str, result_text: str) -> dict:
    response = sync_client.responses.create(
        model=MODEL,
        input=build_input(narrative, result_text),
        text={
            "format": {
                "type": "json_schema",
                "name": "incident_enrichment",
                "schema": JSON_SCHEMA,
                "strict": True,
            }
        },
    )
    return parse_json_output(response.output_text)

async def enrich_one_async(narrative: str, result_text: str) -> dict:
    response = await async_client.responses.create(
        model=MODEL,
        input=build_input(narrative, result_text),
        text={
            "format": {
                "type": "json_schema",
                "name": "incident_enrichment",
                "schema": JSON_SCHEMA,
                "strict": True,
            }
        },
    )
    return parse_json_output(response.output_text)

## Sequential baseline

This runs one incident at a time so you have a baseline for comparison.

In [10]:
def run_sync_pipeline(input_df: pd.DataFrame):
    start = time.perf_counter()
    rows = []

    for _, row in input_df.iterrows():
        enriched = enrich_one_sync(row["Narrative"], row["Result"])
        rows.append(enriched)

    elapsed = time.perf_counter() - start
    enriched_df = input_df.reset_index(drop=True).copy()
    enriched_df["timeline_text"] = [r["timeline_text"] for r in rows]
    enriched_df["normalized_summary"] = [r["normalized_summary"] for r in rows]
    return enriched_df, elapsed

## Async version

This launches one LLM request per incident and waits for them together with `asyncio.gather`.

A semaphore is used to cap concurrency.

In [11]:
async def enrich_row_async(row: pd.Series, semaphore: asyncio.Semaphore) -> dict:
    async with semaphore:
        enriched = await enrich_one_async(row["Narrative"], row["Result"])
        return {
            "timeline_text": enriched["timeline_text"],
            "normalized_summary": enriched["normalized_summary"],
        }

async def run_async_pipeline(input_df: pd.DataFrame, concurrency_limit: int = CONCURRENCY_LIMIT):
    start = time.perf_counter()
    semaphore = asyncio.Semaphore(concurrency_limit)

    tasks = [
        enrich_row_async(row, semaphore)
        for _, row in input_df.iterrows()
    ]

    rows = await asyncio.gather(*tasks)
    elapsed = time.perf_counter() - start

    enriched_df = input_df.reset_index(drop=True).copy()
    enriched_df["timeline_text"] = [r["timeline_text"] for r in rows]
    enriched_df["normalized_summary"] = [r["normalized_summary"] for r in rows]
    return enriched_df, elapsed

## Run both versions and compare timing

In [12]:
sync_enriched_df, sync_elapsed = run_sync_pipeline(demo_df)
async_enriched_df, async_elapsed = asyncio.run(run_async_pipeline(demo_df))

print(f"Sequential processing: {sync_elapsed:.2f} seconds")
print(f"Async processing:      {async_elapsed:.2f} seconds")

if async_elapsed > 0:
    print(f"Speedup:               {sync_elapsed / async_elapsed:.2f}x")

Sequential processing: 500.87 seconds
Async processing:      91.89 seconds
Speedup:               5.45x


## Inspect the enriched fields

In [13]:
preview_cols = ["ACN", "Synopsis", "timeline_text", "normalized_summary"]
async_enriched_df[preview_cols].head(10)

,ACN,Synopsis,timeline_text,normalized_summary
0,2301768,A320 Captain reported rejecting a takeoff at 9...,During takeoff roll approaching 100 kts the cr...,During the takeoff roll the crew rejected take...
1,2301817,B777 crew reported receiving a cargo fire EICA...,Pushback completed with both engines running a...,An aft cargo fire EICAS message occurred after...
2,2302457,Air carrier Captain reported a fuel leak and s...,Pre-departure fuel config light would not stay...,During cruise the crew detected a rapid fuel l...
3,2302774,Air carrier flight crew reported experiencing ...,Shortly after takeoff (800–1000') the throttle...,During initial climb the airspeed bug repeated...
4,2302787,B757 First Officer reported a fuel leak that r...,Flight planned with increased cruise speed to ...,Approximately one hour into the flight the cre...
5,2302908,With an inoperative air speed indicator; the F...,Previous crew performed an air return for unre...,A ferry permit allowed departure with ADR1 ino...
6,2302975,Air carrier Captain reported that while perfor...,After engine start; during the flight control ...,During the preflight flight control check the ...
7,2303029,Air carrier First Officer reported an inflight...,Cruising at FL350 off the coast when a flight ...,A suspected cabin fume/odor event caused multi...
8,2303103,DA40 student pilot reported that the rear door...,At approximately 600 ft AGL on the upwind; ins...,The rear passenger door unlatched and separate...
9,2303193,A321 NEO flight crew reported a rejected takeo...,Attempted takeoff; discontinued takeoff due to...,Crew discontinued takeoff due to reported engi...


## Compare original text with the LLM outputs for one record

In [14]:
row = async_enriched_df.loc[0]

print("ACN:", row["ACN"])
print("\nSYNOPSIS:\n")
print(row["Synopsis"])

print("\nNARRATIVE:\n")
print(row["Narrative"][:2500])

print("\nRESULT:\n")
print(row["Result"][:1500])

print("\nTIMELINE TEXT:\n")
print(row["timeline_text"])

print("\nNORMALIZED SUMMARY:\n")
print(row["normalized_summary"])

ACN: 2301768

SYNOPSIS:

A320 Captain reported rejecting a takeoff at 95 knots due to an engine fire warning and indication on the right engine.  Crew returned to gate after CFR evaluated the aircraft.

NARRATIVE:

Approaching 100 kts (approximately 95 kts); we received a Master Warning and Engine 2 Fire indication. As the reject was initiated; the fire indication disappeared. After completing the Reject maneuver; we referenced the QRC and confirmed with Tower and FAs that they saw no indications of a fire. We then reviewed the Rejected Takeoff checklist in the Flight Manual; exited the runway; and had ARFF (Airport Rescue and Firefighting) look over the aircraft. After ARFF indicted that everything looked normal; we shutdown Engine 2 to taxi back using normal single-engine procedures. After parking and confirming that the aircraft was chocked; the brakes were left off in accordance with flight manual guidance. Throughout the situation; passengers were kept informed and there was no in

## Save the enriched dataframe

This writes the async result to a new CSV so you can use it later in pandas or a dashboard.

In [15]:
async_enriched_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved enriched file to: {OUTPUT_CSV}")

Saved enriched file to: asrs_incidents_enriched.csv


## Notes

- This is a natural async workload because each incident can be processed independently.
- The LLM task is intentionally modest: it **normalizes** the free text instead of recreating the structured ASRS labels already present in the dataset.
- If you want a faster and cheaper demo, lower `MAX_ROWS`.
- If you hit rate limits, lower `CONCURRENCY_LIMIT`.